##DATA2-309 
#Steps
* Re-run collection_process for businessdates: 2025-04-29
2025-04-30
2025-05-02
2025-05-05

## DBTC-1325
# Steps
* Merge the current pull request
* Run the following sql statements in production

In [0]:
env="p"

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.exposure
ADD COLUMN  JustificationCode int AFTER DueDebtAmount;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.exposure
ADD COLUMN  JustificationComments string AFTER JustificationCode ;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryarchive.exposure
ADD COLUMN  JustificationCode int AFTER DueDebtAmount;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryarchive.exposure
ADD COLUMN  JustificationComments string AFTER JustificationCode;
""")

##RHOD-120
#Steps
* Run the following sql 

In [0]:
%sql
MERGE INTO `p-cdh`.derivativeactive.dexposure AS target
USING `qa-cdh`.reference.dexposure_updateGBV_DomusRhodium AS updates
ON target.ExposureDerivativeID = updates.ExposureDerivativeID
   AND target.businessdate = updates.businessdate
WHEN MATCHED THEN
  UPDATE SET target.GBV = updates.GBV;

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)

